In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import pathlib
import glob

import matplotlib as mpl
from matplotlib import rc
from matplotlib.lines import Line2D
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_debug_nans", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODE, NeuralEulerODEPendulum, NeuralEulerODECartpole
from dmpe.evaluation.plotting_utils import plot_sequence, plot_feature_combinations
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.models.model_training import ModelTrainer
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult

In [ ]:
from dmpe.related_work.random_walk import random_walk_control_law
from plot_helpers import plot_jsd_model_prediction_relation, plot_model_rollouts

In [ ]:
from dmpe.utils.sets.shared import DiscretizedSet, load_discretized_set, check_in_set 
from dmpe.utils.density_estimation import build_grid

In [ ]:
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
S_xu = load_discretized_set(
    DataPaths().reach_ci_experiments / "cart_pole_S_xu_8744b5d5-30e6-4b.json",
)

In [ ]:
env, penalty_function, featurize, _ = setup_cart_pole_env()

In [ ]:
check_in_S_xu = partial(check_in_set, grid=S_xu.grid, mask=S_xu.mask)
constraint_function = lambda z: jnp.logical_not(check_in_S_xu(z))
# jax.vmap(constraint_function, in_axes=0)(build_grid(5, -1, 1, 5))

In [ ]:
model_evaluator = ModelEvaluator(
    constraint_function=constraint_function,
    gt_model=EnvWrapper(env, featurize=featurize),
    obs_dim=4,
    act_dim=1,
    validation_points_per_dim=20,
    tau=env.tau,
)
data_evaluator = DataEvaluator(
    constraint_function=constraint_function,
    data_dim=5,
    points_per_dim=20,
)

In [ ]:
def plot_jsd_model_prediction_relation_with_model_and_data_evaluator(
    data_path: pathlib.Path,
    model_class: eqx.Module,
    model_evaluator: ModelEvaluator,
    data_evaluator: DataEvaluator,
    verbose: bool = False,
    expecting_sub_folders: bool = True,
):
    means = []
    medians = []
    jsds = []
    colors = []

    color_cycle = plt.rcParams["axes.prop_cycle"]()
    color_mapping = [next(color_cycle)["color"] for _ in range(15)]

    result_paths = (
        glob.glob(str(data_path) + "/**/*.eqx") if expecting_sub_folders else glob.glob(str(data_path) + "/*.eqx")
    )

    n_results = len(result_paths)
    print("# or results:", n_results)
    print(80 * "-")

    for result_path in tqdm(result_paths, total=len(result_paths)):
        result = ModelExpDataResult.from_file(
            filename=result_path,
            model_class=model_class,
        )

        color_idx = int(result.n_datapoints / 1_000) - 1
        colors.append(color_mapping[color_idx])

        model_error = model_evaluator.default_metrics["pred_comp"](
            NodeModelWrapper(result.median_model, featurize=featurize), model_evaluator.gt_model
        )[1]
        medians.append(model_error)

        jsd_value = data_evaluator.get_metrics(
            data_points=jnp.concatenate([result.observations, result.actions], axis=-1)
        )["jsd"]

        jsds.append(jsd_value)
        if verbose:
            print(result.data_jsd)
            fig, _ = result.visualize()
            plt.show()
            print(80 * "-")

    fig, ax = plt.subplots(1, 1, figsize=(12, 8))

    ax.scatter(
        jsds, medians, s=25, marker="x", c=colors
    )  # , c=next(colors)["color"], label=f"{data_length} data points")

    ax.set_ylabel("model prediction loss")
    ax.set_xlabel("JSD")
    # ax.set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
    # ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
    ax.grid(True)
    ax.set_yscale("log")
    ax.set_xscale("log")
    legend_elements = [
        Line2D(
            [0],
            [0],
            marker="x",
            color="w",
            label=(idx + 1) * 1000,
            markerfacecolor=color_mapping[idx],
            markeredgecolor=color_mapping[idx],
            markersize=5,
            linestyle="None",
        )
        for idx in range(len(color_mapping))
    ]
    # legend_elements = [Patch(facecolor=color_mapping[idx], label=(idx + 1) * 1000) for idx in range(len(color_mapping))]
    ax.legend(handles=legend_elements, title=r"\# of datapoints")

    return fig, ax

In [ ]:
data_evaluator.default_metrics

In [ ]:
# investigate example for one single result:
data_path = DataPaths().model_learning_cs_out / "cart_pole" / "2step"
result_path = glob.glob(str(data_path) + "/*.eqx")[1]

result = ModelExpDataResult.from_file(
    filename=result_path,
    model_class=NeuralEulerODECartpole,
)

# actually plot it

In [ ]:
fig, axs = plot_jsd_model_prediction_relation_with_model_and_data_evaluator(
    DataPaths().model_learning_cs_out / "cart_pole" / "2step",
    model_class=NeuralEulerODECartpole,
    model_evaluator=model_evaluator,
    data_evaluator=data_evaluator,
    verbose=False,
    expecting_sub_folders=False,
)
plt.savefig("test.png")

In [ ]:
plot_jsd_model_prediction_relation(
    data_path=DataPaths().model_learning_cs_out / "cart_pole" / "2step",
    model_class=NeuralEulerODECartpole,
    verbose=False,
    expecting_sub_folders=False,
    recompute_jsd=True,
)